# DYNAMIC METADATA GENERATION

This script automatically generates metadata from CSV folders
located in the Databricks volume.
It replaces manual INSERT statements by scanning directories
and dynamically building the metadata configuration.

## metadata.tables

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, TimestampType, LongType

base_path = "/Volumes/banking/source/volume/" 

# List all files from the source volume
files = dbutils.fs.ls(base_path)

tables = []

# Business load order mapping
load_order_mapping = {
    "customers": 1,
    "accounts": 2,
    "transactions": 3,
    "branches": 4,
    "credit_bureau_reports": 5,
    "payment_gateway_logs": 6
}

for f in files:
    if f.name.endswith(".csv"):
        table_name = f.name.replace(".csv", "")
        if table_name in load_order_mapping:
            tables.append((
                load_order_mapping[table_name],   # table_id
                table_name,                       
                "csv",                            # source_system
                None,                             # source_schema
                None,                             # source_table
                f.path,                           # source_path
                "silver",                         # target_layer
                "bronze",                         # bronze_schema
                "silver",                         # silver_schema
                None,                             # gold_schema
                True,                             # active_flag
                load_order_mapping[table_name],   # load_order
                datetime.now()                    # created_at
            ))

# Metadata table columns
columns = [
    "table_id",
    "table_name",
    "source_system",
    "source_schema",
    "source_table",
    "source_path",
    "target_layer",
    "bronze_schema",
    "silver_schema",
    "gold_schema",
    "active_flag",
    "load_order",
    "created_at"
]
# We use StructType to explicitly define the schema so Spark does not need to infer data types,
# because the data contains None values that can cause schema inference errors.

schema = StructType([
    StructField("table_id", IntegerType(), True),
    StructField("table_name", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_schema", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("target_layer", StringType(), True),
    StructField("bronze_schema", StringType(), True),
    StructField("silver_schema", StringType(), True),
    StructField("gold_schema", StringType(), True),
    StructField("active_flag", BooleanType(), True),
    StructField("load_order", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

df = spark.createDataFrame(tables, schema)

# Insert metadata into Delta table
df.write.mode("append").saveAsTable("banking.metadata.tables")
display(df)

## LOAD METADATA PARAMETERS GENERATION 
## metadata.table_parameters
This section generates configurable ETL parameters for each table
including:
- load type (MERGE / APPEND / FULL)
- primary key definition
- watermark column for incremental processing

Summary per table
- customers → MERGE (update customers using updated_at)
- accounts → MERGE (update accounts using updated_at)
- transactions → APPEND (keep all transactions using txn_timestamp)
- branches → FULL (reload everything, no incremental logic)


In [0]:
# Metadata rules
rules = {
    "customers": ("MERGE", "customer_id", "updated_at"),
    "accounts": ("MERGE", "account_id", "updated_at"),
    "transactions": ("APPEND", "txn_id", "txn_timestamp"),
    "branches": ("FULL", "branch_code", None),
    "credit_bureau_reports": ("MERGE", "customer_id", "bureau_pull_date"),
    "payment_gateway_logs": ("APPEND", "txn_id", "processed_timestamp")
}

params = []
for table_name, table_id in load_order_mapping.items():
    if table_name in rules:
        load_type, primary_key, watermark_column = rules[table_name]
        params.append((
            table_id,
            "load_type",
            load_type,
            datetime.now()
        ))
        params.append((
            table_id,
            "primary_key",
            primary_key,
            datetime.now()
        ))
        if watermark_column:
            params.append((
                table_id,
                "watermark_column",
                watermark_column,
                datetime.now()
            ))

df = spark.createDataFrame(params, [
    "table_id",
    "parameter_name",
    "parameter_value",
    "created_at"
])

# Write metadata into Delta table
df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("banking.metadata.table_parameters")
display(df)


## INITIAL WATERMARK INITIALIZATION
This section initializes the watermark tracking table used by incremental ingestion pipelines.
It stores the last processed timestamp for each table to ensure that historical data is not reprocessed.

FULL tables are excluded from watermark tracking because they are fully reloaded on each run.
Since the entire dataset is replaced during every execution (no incremental logic), there is no need to maintain a last processed value.

Watermarks are only applied to incremental loads (MERGE or APPEND) to efficiently process only new or updated records.

In [0]:
incremental_tables = [
    table_name
    for table_name, (load_type, _, _) in rules.items()
    if load_type != "FULL"
]
watermarks = []

for table_name, table_id in load_order_mapping.items():

    if table_name in incremental_tables:

        watermarks.append((
            table_id,
            "1900-01-01 00:00:00",   # initial watermark
            datetime.now(),          # last updated
            None                      # last run id
        ))

schema = StructType([
    StructField("table_id", IntegerType(), True),
    StructField("last_watermark_value", StringType(), True),
    StructField("last_updated_at", TimestampType(), True),
    StructField("last_run_id", LongType(), True)
])

df = spark.createDataFrame(watermarks, schema)

df.write.mode("overwrite").saveAsTable(
    "banking.metadata.table_watermarks"
)
display(df)

## GOLD LAYER TABLE METADATA DEFINITION
This section defines business-level (Gold) tables used for analytics and reporting.
These tables are built from Silver layer data and represent aggregated or business-ready datasets.

In [0]:
gold_tables = [
    (7, "customer_profile", 1),
    (8, "branch_performance", 2),
    (9, "transaction_channel_summary", 3),
    (10, "daily_bank_kpi", 4),
    (11, "risk_customer_summary", 1)
]

gold_rows = []

for table_id, table_name, load_order in gold_tables:
    gold_rows.append((
        table_id,
        table_name,
        "silver",
        None,
        None,
        None,
        "gold",
        None,
        None,
        "gold",
        True,
        load_order,
        datetime.now()
    ))
    
schema = StructType([
    StructField("table_id", IntegerType(), True),
    StructField("table_name", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_schema", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("target_layer", StringType(), True),
    StructField("bronze_schema", StringType(), True),
    StructField("silver_schema", StringType(), True),
    StructField("gold_schema", StringType(), True),
    StructField("active_flag", BooleanType(), True),
    StructField("load_order", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

df_gold = spark.createDataFrame(gold_rows, schema)

df_gold.write.mode("append").saveAsTable(
    "banking.metadata.tables"
)

display(df_gold)